# PathSnap HMM and Viterbi
Inference-only state is a candidate arc at each GPS time. The transition log potential is negative absolute error between directed travel distance and GPS displacement, divided by 20 m plus 0.25 times the displacement. Travel distance uses projected partial arcs and a bounded directed shortest path. The hard distance bound is 20 m plus 55 m/s times elapsed time. Parameters were set before student-test evaluation and were not fitted. Scores are clipped and normalised within valid candidates, then logged. Viterbi maximises previous path score plus transition plus emission. Impossible transitions have log score negative infinity. Large gaps, missing states and wholly unreachable rows break the sequence. Inference receives neither reference arcs nor labels.

In [1]:
from dataclasses import dataclass, asdict
from functools import lru_cache
from heapq import heappop, heappush
from pathlib import Path
from time import perf_counter
import csv, json, os, platform
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

@dataclass(frozen=True)
class DecoderConfig:
    score_floor: float = 1e-12
    base_distance_scale_m: float = 20.
    displacement_scale: float = .25
    maximum_speed_mps: float = 55.
    positional_slack_m: float = 20.
    restart_gap_s: float = 30.
    distance_cache_size: int = 4096

class RoadGraph:
    def __init__(self, nodes_xy, arcs):
        start = perf_counter()
        self.nodes = np.asarray(nodes_xy, dtype=float)
        self.arcs = np.asarray(arcs)
        if self.nodes.ndim != 2 or self.nodes.shape[1] != 2 or not np.isfinite(self.nodes).all():
            raise ValueError("Nodes must be a finite N × 2 array in metres")
        if (self.arcs.ndim != 2 or self.arcs.shape[1] != 2
                or not np.issubdtype(self.arcs.dtype, np.integer)
                or (self.arcs < 0).any() or (self.arcs >= len(self.nodes)).any()):
            raise ValueError("Invalid directed arc endpoints")
        self.start, self.end = self.arcs.T
        self.origin = self.nodes[self.start]
        self.vector = self.nodes[self.end] - self.origin
        self.length = np.linalg.norm(self.vector, axis=1)
        self.outgoing = [[] for _ in self.nodes]
        for a, b, length in zip(self.start, self.end, self.length):
            self.outgoing[int(a)].append((int(b), float(length)))
        topology = csr_matrix((np.ones(len(self.arcs)), (self.start,self.end)),
                              shape=(len(self.nodes),len(self.nodes)))
        self.weak_components, self.component = connected_components(topology, directed=True, connection="weak")
        self.build_seconds = perf_counter() - start

    def bounded_distances(self, source, limit):
        distances = {source: 0.}
        queue = [(0., source)]
        while queue:
            distance, node = heappop(queue)
            if distance != distances[node]:
                continue
            for endpoint, weight in self.outgoing[node]:
                length = distance + weight
                if length <= limit and length < distances.get(endpoint, np.inf):
                    distances[endpoint] = length
                    heappush(queue, (length, endpoint))
        return distances

def emission_scores(arc_ids, scores, arc_count, floor):
    if arc_ids.shape != scores.shape or arc_ids.ndim != 2 or not np.issubdtype(arc_ids.dtype,np.integer):
        raise ValueError("Candidate IDs and scores must have the same N × K shape")
    valid = (arc_ids >= 0) & (arc_ids < arc_count) & np.isfinite(scores) & (scores >= 0) & (scores <= 1)
    mass = np.where(valid, np.maximum(scores, floor), 0.)
    total = np.sum(mass,axis=1,keepdims=True)
    logs = np.full_like(mass,-np.inf,dtype=float)
    np.log(np.divide(mass,total,out=np.zeros_like(mass,dtype=float),where=total>0),out=logs,where=valid)
    return logs, valid

def projected_offsets(point, graph, ids):
    vector = graph.vector[ids]
    fraction = np.einsum("ij,ij->i", point - graph.origin[ids], vector)
    lengths = graph.length[ids]
    return np.clip(np.divide(fraction,lengths**2,out=np.zeros_like(fraction),where=lengths>0),0,1)*lengths

def viterbi_decode(trip_id, xy, times, arc_ids, scores, graph, config=DecoderConfig()):
    """Inference-only decoder. Returns one aligned arc ID per point (-1 = no valid candidate)."""
    start = perf_counter()
    xy = np.asarray(xy,dtype=float)
    times = np.asarray(times,dtype=float)
    ids = np.asarray(arc_ids)
    scores = np.asarray(scores,dtype=float)
    if (xy.shape != (len(times),2) or times.ndim != 1 or len(times)==0 or len(ids)!=len(times)
            or not np.isfinite(xy).all() or not np.isfinite(times).all() or (np.diff(times)<0).any()
            or ids.ndim!=2 or ids.shape[1]==0):
        raise ValueError("Expected nonempty ordered finite GPS points and aligned candidates")
    if (not 0 < config.score_floor <= 1 or config.base_distance_scale_m <= 0
            or config.displacement_scale < 0 or config.maximum_speed_mps <= 0
            or config.positional_slack_m < 0 or config.restart_gap_s <= 0
            or config.distance_cache_size < 0):
        raise ValueError("Invalid decoder configuration")
    emissions, valid = emission_scores(ids,scores,len(graph.arcs),config.score_floor)
    n, k = ids.shape
    offsets = np.vstack([projected_offsets(p,graph,np.where(ok,row,0))
                         for p,row,ok in zip(xy,ids,valid)]) if len(graph.arcs) else np.zeros_like(scores)
    @lru_cache(maxsize=config.distance_cache_size)
    def distances(source, bucket):
        return graph.bounded_distances(source, bucket)

    def costs(t):
        gap = times[t]-times[t-1]
        step = np.linalg.norm(xy[t]-xy[t-1])
        limit = config.positional_slack_m + config.maximum_speed_mps*gap
        # Round upward only for caching; filter actual path distances against the exact limit.
        bucket = int(np.ceil(limit/100))*100
        matrix = np.full((k,k),np.inf)
        for i in np.flatnonzero(valid[t-1]):
            a = int(ids[t-1,i])
            end = int(graph.end[a])
            remainder = graph.length[a] - offsets[t-1,i]
            to_nodes = distances(end,bucket)
            for j in np.flatnonzero(valid[t]):
                b = int(ids[t,j])
                begin = int(graph.start[b])
                direct = offsets[t,j]-offsets[t-1,i] if a==b and offsets[t,j]>=offsets[t-1,i] else np.inf
                routed = remainder + to_nodes.get(begin,np.inf) + offsets[t,j]
                distance = min(direct,routed)
                if distance <= limit:
                    matrix[i,j] = distance
        scale = config.base_distance_scale_m + config.displacement_scale*step
        return -np.abs(matrix-step)/scale

    back = np.full((n,k),-1,dtype=np.int32)
    end_scores = np.full((n,k),-np.inf)
    breaks = {}
    classifier = np.argmax(emissions,axis=1)
    base_arc = np.where(valid.any(axis=1),ids[np.arange(n),classifier],-1)
    baseline_feasible = np.zeros(max(0,n-1),dtype=bool)
    valid_pair = np.zeros(max(0,n-1),dtype=bool)
    for t in range(n):
        if not valid[t].any():
            breaks[t] = "missing_candidates"
            continue
        if t==0:
            end_scores[t]=emissions[t]
            continue
        if not valid[t-1].any():
            breaks[t]="after_missing"
        elif times[t]-times[t-1]>config.restart_gap_s:
            breaks[t]="large_gap"
        else:
            transition=costs(t)
            valid_pair[t-1]=True
            baseline_feasible[t-1]=np.isfinite(transition[classifier[t-1],classifier[t]])
            previous=end_scores[t-1,:,None]+transition
            pointers=np.argmax(previous,axis=0)
            best=previous[pointers,np.arange(k)] + emissions[t]
            if np.isfinite(best).any():
                end_scores[t]=best
                back[t]=np.where(np.isfinite(best),pointers,-1)
                continue
            breaks[t]="no_feasible_transition"
        end_scores[t]=emissions[t]

    chosen = np.full(n,-1,dtype=np.int64)
    for t in range(n-1,-1,-1):
        if chosen[t]<0 and np.isfinite(end_scores[t]).any():
            chosen[t]=int(np.argmax(end_scores[t]))
        if t>0 and chosen[t]>=0 and back[t,chosen[t]]>=0:
            chosen[t-1]=back[t,chosen[t]]
    predicted=np.where(chosen>=0,ids[np.arange(n),np.maximum(chosen,0)],-1)
    hmm_feasible=(chosen[:-1]>=0)&(chosen[1:]>=0)&(back[np.arange(1,n),np.maximum(chosen[1:],0)]>=0)
    elapsed=perf_counter()-start
    return {"trip_id":str(trip_id),"arc_ids":predicted,"candidate_indices":chosen,
            "classifier_arc_ids":base_arc,"breaks":breaks,"baseline_feasible":baseline_feasible,
            "hmm_feasible":hmm_feasible,"eligible_pairs":valid_pair,
            "decode_seconds":elapsed,"distance_cache":distances.cache_info()._asdict(),
            "dp_array_bytes":back.nbytes+end_scores.nbytes+offsets.nbytes+emissions.nbytes}



## Synthetic correctness tests
These deterministic unit checks exercise the decoder without external files.

In [2]:
# Synthetic correctness tests. These are not experiment scores.
def run_synthetic_tests():
    graph=RoadGraph([[0,0],[10,0],[20,0],[30,0],[10,10],[20,10]],
                    np.array([[0,1],[1,2],[2,3],[4,5]]))
    cfg=DecoderConfig(maximum_speed_mps=20,positional_slack_m=2)
    def decode(x,t,ids,s):
        return viterbi_decode("synthetic",x,t,np.array(ids),np.array(s),graph,cfg)
    chain=decode([[5,0],[15,0],[25,0]],[0,1,2],[[0,3],[1,3],[2,3]],
                 [[.9,.1],[.2,.8],[.9,.1]])
    assert chain["classifier_arc_ids"].tolist()==[0,3,2]
    assert chain["arc_ids"].tolist()==[0,1,2] and chain["hmm_feasible"].all()
    intermediate=decode([[5,0],[25,0]],[0,1],[[0],[2]],[[1.],[1.]])
    assert intermediate["hmm_feasible"].tolist()==[True]
    reverse=decode([[25,0],[5,0]],[0,1],[[2],[0]],[[1.],[1.]])
    assert reverse["breaks"]=={1:"no_feasible_transition"} and reverse["arc_ids"].tolist()==[2,0]
    assert decode([[5,0],[15,0]],[0,100],[[0],[1]],[[1.],[1.]])["breaks"]=={1:"large_gap"}
    assert decode([[5,0],[15,0]],[0,0],[[0],[1]],[[1.],[1.]])["breaks"]=={1:"no_feasible_transition"}
    assert decode([[5,0],[5,0]],[0,0],[[0],[0]],[[1.],[1.]])["hmm_feasible"].all()
    missing=decode([[5,0],[5,0],[5,0]],[0,1,2],[[0],[-1],[0]],[[1.],[np.nan],[1.]])
    assert missing["arc_ids"].tolist()==[0,-1,0] and missing["breaks"]=={1:"missing_candidates",2:"after_missing"}
    assert decode([[5,0]],[0],[[0,1]],[[0.,1e-100]])["arc_ids"].tolist()==[0]
    repeat=decode([[1,0],[4,0],[7,0]],[0,1,2],[[0],[0],[0]],[[1.],[1.],[1.]])
    assert repeat["arc_ids"].tolist()==[0,0,0] and repeat["hmm_feasible"].all()
    try:
        RoadGraph([[0,0]],np.array([[0,2]]))
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid arc accepted")
    print("Synthetic tests passed: connected alternatives and backtracking, intermediate arcs, one-way/disconnected paths, large/zero GPS gaps, missing states, tiny scores, single/repeated arcs, invalid endpoints.")
run_synthetic_tests()



Synthetic tests passed: connected alternatives and backtracking, intermediate arcs, one-way/disconnected paths, large/zero GPS gaps, missing states, tiny scores, single/repeated arcs, invalid endpoints.


## Student-test experiment
Use the saved tuned Extra Trees probabilities for the same 19,160 ordered points in both runs. All observations count in Top-1. Report exact shared trip/arc Macro F1, plus the fraction of consecutive pairs on the same arc or sharing a directed endpoint and the fraction admitting a directed path under the speed rule without a break. Denominators include all within-trip adjacent pairs, so large gaps count as breaks. Sequence similarity is one minus normalised Levenshtein distance on run-collapsed matched arc IDs against the aligned proxy labels. This is separate from comparison against the complete supplied reference traversal. Viterbi outputs paths rather than calibrated state marginals, so categorical log loss is not applied to its hard decisions.

In [3]:
root=Path.cwd()
if root.name=="notebooks":
    root=root.parent
os.chdir(root)
get_ipython().run_line_magic("run","notebooks/00_shared_training.ipynb")
import scipy
from collections import Counter

processed=Path("processed/student_test")
raw=Path("Student folder/student_test")
model_dir=Path("results/bayesian_extra_trees_01")
prediction_file=model_dir/"tuned_student_test_predictions.npz"
output=Path("results/hmm_viterbi_01")
output.mkdir(parents=True,exist_ok=True)
config=DecoderConfig()

def load_inference_trip(trip):
    with np.load(processed/f"{trip}.npz",allow_pickle=False) as d:
        prepared={k:d[k] for k in ("xy_m","gps","original_row","projection_origin_lonlat","candidate_arc_ids")}
    folder=raw/trip
    nodes=np.loadtxt(folder/f"{trip}.nodes",ndmin=2)
    arcs=np.loadtxt(folder/f"{trip}.arcs",dtype=np.int64,ndmin=2)
    lon,lat=prepared["projection_origin_lonlat"]
    nodes_xy=6371008.8*np.deg2rad(nodes-[lon,lat])*[np.cos(np.deg2rad(lat)),1]
    return prepared, nodes, arcs, RoadGraph(nodes_xy,arcs)

def collapsed(seq):
    return seq[np.r_[True,seq[1:]!=seq[:-1]]]

def edit_similarity(a,b):
    # Unit-cost insertion/deletion/substitution on run-collapsed trip-local arc IDs.
    a,b=collapsed(a),collapsed(b)
    prior=list(range(len(b)+1))
    b=b.tolist()
    for i,x in enumerate(a.tolist(),1):
        cur=[i]
        for j,y in enumerate(b,1):
            cur.append(min(cur[-1]+1,prior[j]+1,prior[j-1]+(x!=y)))
        prior=cur
    return 1-prior[-1]/max(len(a),len(b),1)

def direct_links(sequence,graph):
    a,b=sequence[:-1],sequence[1:]
    return (a>=0)&(b>=0)&((a==b)|(graph.end[np.maximum(a,0)]==graph.start[np.maximum(b,0)]))

def hard_decision_metrics(data,ids,sequence):
    # Reuse the exact shared trip-and-arc aggregation by passing one-hot decoded choices.
    hits=ids==sequence[:,None]
    if np.any(~hits.any(axis=1)):
        # Preserve absent rows with the explicit -1 abstention state.
        data=dict(data)
        data["candidate_arc_ids"]=np.column_stack([ids,np.full(len(ids),-1)])
        data["y"]=np.column_stack([data["y"],np.zeros(len(ids),dtype=np.int8)])
        hits=np.column_stack([hits,sequence<0])
    scores=hits.astype(float)
    metric,_=score_predictions(data,scores)
    def matching_only(m):
        return {k:v for k,v in m.items() if "log_loss" not in k} if m else None
    return dict(all_points=matching_only(metric["all_points"]),
                reliable_points=matching_only(metric["reliable_points"]),
                reliable_coverage=metric["reliable_coverage"])

with np.load(prediction_file,allow_pickle=False) as file:
    saved={k:file[k] for k in file.files}
assert saved["positive_probability"].shape==(19160,8)
trip_ids=np.unique(saved["trip_id"])
assert len(trip_ids)==10 and set(trip_ids)=={p.stem for p in processed.glob("*.npz")}

trip=trip_ids[0]
p,_,_,g=load_inference_trip(trip)
sel=saved["trip_id"]==trip
np.testing.assert_array_equal(p["candidate_arc_ids"],saved["candidate_arc_ids"][sel])
smoke=viterbi_decode(trip,p["xy_m"][:40],p["gps"][:40,2],p["candidate_arc_ids"][:40],
                     saved["positive_probability"][sel][:40],g,config)
assert len(smoke["arc_ids"])==40
print("Real-data smoke check passed for 40 GPS observations. Scores below use the full student-test set.")

rows=[]; all_baseline=[]; all_decoded=[]; all_original=[]; network={}; all_breaks=[]
for trip in trip_ids:
    idx=saved["trip_id"]==trip
    p,nodes,arcs,graph=load_inference_trip(trip)
    ids=saved["candidate_arc_ids"][idx]
    np.testing.assert_array_equal(ids,p["candidate_arc_ids"])
    result=viterbi_decode(trip,p["xy_m"],p["gps"][:,2],ids,saved["positive_probability"][idx],graph,config)
    baseline=result["classifier_arc_ids"]
    decoded=result["arc_ids"]
    np.testing.assert_array_equal(baseline,saved["predicted_arc_id"][idx])
    # Reference arrays are loaded and used only after decoder inference.
    with np.load(processed/f"{trip}.npz",allow_pickle=False) as dataset:
        truth=dataset["reference_arc_id"]
        np.testing.assert_array_equal(truth,saved["reference_arc_id"][idx])
        np.testing.assert_array_equal(dataset["label_reliable"],saved["label_reliable"][idx])
        data={"y":dataset["y"],"reference_arc_id":truth,"trip_id":saved["trip_id"][idx],
              "label_reliable":dataset["label_reliable"],"candidate_arc_ids":ids}
    bm=hard_decision_metrics(data,ids,baseline)
    hm=hard_decision_metrics(data,ids,decoded)
    n_pairs=len(ids)-1
    meta=dict(nodes=len(nodes),arcs=len(arcs),weak_components=int(graph.weak_components),
              build_seconds=graph.build_seconds,decode_seconds=result["decode_seconds"],
              dp_array_bytes=result["dp_array_bytes"],distance_cache=result["distance_cache"])
    network[str(trip)]=meta
    for t,reason in result["breaks"].items():
        all_breaks.append(dict(trip_id=str(trip),observation=int(t),original_row=int(p["original_row"][t]),reason=reason))
    row=dict(trip_id=str(trip),observations=len(ids),pairs=n_pairs,
             baseline_top1=bm["all_points"]["top1_accuracy"],hmm_top1=hm["all_points"]["top1_accuracy"],
             baseline_macro_f1=bm["all_points"]["macro_f1"],hmm_macro_f1=hm["all_points"]["macro_f1"],
             baseline_direct=int(direct_links(baseline,graph).sum()),hmm_direct=int(direct_links(decoded,graph).sum()),
             baseline_feasible=int(result["baseline_feasible"].sum()),hmm_feasible=int(result["hmm_feasible"].sum()),
             eligible_pairs=int(result["eligible_pairs"].sum()),baseline_edit_similarity=edit_similarity(baseline,truth),
             hmm_edit_similarity=edit_similarity(decoded,truth),breaks=len(result["breaks"]),
             decode_seconds=result["decode_seconds"])
    rows.append(row)
    all_baseline.append(baseline);all_decoded.append(decoded);all_original.append(p["original_row"])
    print(f"{trip} {len(ids)} observations: Top-1 {row['baseline_top1']:.3%} → {row['hmm_top1']:.3%}, "
          f"motion-feasible {row['baseline_feasible']}/{n_pairs} → {row['hmm_feasible']}/{n_pairs}, "
          f"decode {row['decode_seconds']:.2f}s",flush=True)
baseline=np.concatenate(all_baseline);decoded=np.concatenate(all_decoded)
data=dict(y=(saved["candidate_arc_ids"]==saved["reference_arc_id"][:,None]).astype(np.int8),
          candidate_arc_ids=saved["candidate_arc_ids"],reference_arc_id=saved["reference_arc_id"],
          trip_id=saved["trip_id"],label_reliable=saved["label_reliable"])
metrics={"classifier":hard_decision_metrics(data,data["candidate_arc_ids"],baseline),
         "hmm":hard_decision_metrics(data,data["candidate_arc_ids"],decoded)}
pair_count=sum(row["pairs"] for row in rows)
for label,prefix in [("classifier","baseline"),("hmm","hmm")]:
    result=metrics[label]
    result["direct_adjacency_rate"]=sum(row[f"{prefix}_direct"] for row in rows)/pair_count
    result["motion_feasible_rate"]=sum(row[f"{prefix}_feasible"] for row in rows)/pair_count
    result["mean_trip_edit_similarity"]=float(np.mean([row[f"{prefix}_edit_similarity"] for row in rows]))
    result["pairs_not_motion_feasible"]=pair_count-sum(row[f"{prefix}_feasible"] for row in rows)
metrics["diagnostics"]={"pairs":pair_count,"breaks_by_reason":dict(Counter(x["reason"] for x in all_breaks)),
                        "decode_seconds":sum(x["decode_seconds"] for x in rows),
                        "graph_build_seconds":sum(x["build_seconds"] for x in network.values()),
                        "peak_dp_array_bytes":max(x["dp_array_bytes"] for x in network.values())}
for name,obj in {"config":asdict(config),"metrics":metrics,"networks":network,"breaks":all_breaks,
                 "provenance":{"classifier":"Bayesian Extra Trees trial 5", "numpy":np.__version__,
                               "scipy":scipy.__version__,"python":platform.python_version(),
                               "partition":"student_test","validation_folds":"no saved tuned OOF predictions"}}.items():
    (output/f"{name}.json").write_text(json.dumps(obj,indent=2)+"\n")
with (output/"per_trip.csv").open("w",newline="") as f:
    writer=csv.DictWriter(f,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)
np.savez_compressed(output/"decoded_observations.npz",trip_id=saved["trip_id"],
                    original_row=np.concatenate(all_original),candidate_arc_ids=saved["candidate_arc_ids"],
                    classifier_arc_id=baseline,decoded_arc_id=decoded)
print(json.dumps(metrics,indent=2))



Real-data smoke check passed for 40 GPS observations. Scores below use the full student-test set.
{
  "classifier": {
    "all_points": {
      "observations": 19160,
      "top1_accuracy": 0.811795407098121,
      "macro_precision": 0.6771908416177109,
      "macro_recall": 0.6191479814618747,
      "macro_f1": 0.6357865879035448,
      "candidate_miss_rate": 0.003496868475991649
    },
    "reliable_points": {
      "observations": 13307,
      "top1_accuracy": 0.8868264823025476,
      "macro_precision": 0.786866532998101,
      "macro_recall": 0.7427360114782171,
      "macro_f1": 0.7583572687572395,
      "candidate_miss_rate": 0.0005260389268805891
    },
    "reliable_coverage": 0.6945198329853862,
    "direct_adjacency_rate": 0.8966579634464752,
    "motion_feasible_rate": 0.8693994778067885,
    "mean_trip_edit_similarity": 0.48560387666665966,
    "pairs_not_motion_feasible": 2501
  },
  "hmm": {
    "all_points": {
      "observations": 19160,
      "top1_accuracy": 0.759916

## Geographic and aggregate visualisation
Layered development map draws only the actual geometry of selected arcs. The underlying decoder can use intermediate graph segments, which the plotted selected-arc layer does not interpolate.

In [4]:
import matplotlib.pyplot as plt
import folium
plots=output/"plots"
plots.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi":125,"axes.spines.top":False,"axes.spines.right":False})
def plot_finish(fig,name):
    fig.tight_layout()
    fig.savefig(plots/f"{name}.png")
    plt.close(fig)
fig,axes=plt.subplots(1,4,figsize=(11,3))
for ax,(title,metric,inner) in zip(axes,[("Top-1","top1_accuracy",True),("Macro F1","macro_f1",True),
                                        ("Direct adjacency","direct_adjacency_rate",False),
                                        ("Motion-feasible","motion_feasible_rate",False)]):
    values=[(metrics[k]["all_points"] if inner else metrics[k])[metric] for k in ["classifier","hmm"]]
    ax.bar(["Classifier","HMM"],values,color=["#35668f","#007d68"])
    for i,v in enumerate(values):ax.text(i,v+.02,f"{v:.3f}",ha="center")
    ax.set(ylim=(0,1.06),title=title)
plot_finish(fig,"comparison")
fig,(ax,bx)=plt.subplots(2,1,figsize=(9,6),sharex=True)
positions=np.arange(len(rows))
deltas=[100*(r["hmm_top1"]-r["baseline_top1"]) for r in rows]
ax.bar(positions,deltas,color=["#007d68" if v>=0 else "#b34344" for v in deltas])
ax.axhline(0,color="black",linewidth=.7);ax.set(ylabel="Top-1 change (pp)")
bx.bar(positions,[r["decode_seconds"] for r in rows],color="#35668f")
bx.set(ylabel="Decode seconds",xticks=positions,xticklabels=[r["trip_id"][-4:] for r in rows],xlabel="Trip ID suffix")
plot_finish(fig,"per_trip")
trip=max(rows,key=lambda row:row["hmm_feasible"]-row["baseline_feasible"])["trip_id"]
idx=saved["trip_id"]==trip
p,nodes,arcs,_=load_inference_trip(trip)
gps=p["gps"]
m=folium.Map(location=[float(gps[:,1].mean()),float(gps[:,0].mean())],zoom_start=13)
def road_layer(name,seq,color,weight,show):
    layer=folium.FeatureGroup(name=name,show=show)
    # Draw actual network arc geometry, without fabricating straight connectors.
    for arc in np.unique(seq):
        if arc>=0:
            folium.PolyLine(nodes[arcs[arc]][:,[1,0]].tolist(),color=color,weight=weight).add_to(layer)
    layer.add_to(m)
road_layer("Candidate road network",saved["candidate_arc_ids"][idx].ravel(),"#969696",1,False)
road_layer("Classifier selected arcs",baseline[idx],"#ef8d22",4,False)
road_layer("Viterbi selected arcs",decoded[idx],"#009d69",4,True)
reference=np.loadtxt(raw/trip/f"{trip}.route",dtype=np.int64,ndmin=1)
road_layer("Supplied reference traversal (development only)",reference,"#7a48bc",3,False)
points=folium.FeatureGroup(name="GPS observations",show=True)
for lon,lat,_ in gps:
    folium.CircleMarker([float(lat),float(lon)],radius=2,color="#296ac4",fill=True).add_to(points)
points.add_to(m)
folium.LayerControl().add_to(m)
m.fit_bounds([[float(gps[:,1].min()),float(gps[:,0].min())],[float(gps[:,1].max()),float(gps[:,0].max())]])
m.save(str(output/f"student_trip_{trip}.html"))
print("Saved metrics, per-trip plots, aligned predictions and development map.")


Saved metrics, per-trip plots, aligned predictions and development map.
